# Модель asr из репо

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import shutil

drive_repo_path = "/content/drive/MyDrive/GigaAM_repo"
repo_path = "/content/GigaAM"
shutil.copytree(drive_repo_path, repo_path, dirs_exist_ok=True)
%cd {repo_path}
!pip install -e .

print("Библиотека восстановлена")

/content/GigaAM
Obtaining file:///content/GigaAM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 220.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 200.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.9 MB/s eta 0:00:00
  Building editable for gi

In [3]:
import gigaam
import time

print("Загружаем GigaAM-e2e_ctc...")
start_load = time.time()
model_asr = gigaam.load_model("e2e_ctc").float()
print(f"Загружена за {time.time() - start_load:.2f} сек")

Загружаем GigaAM-e2e_ctc...


100%|███████████████████████████████████████| 422M/422M [00:22<00:00, 20.1MiB/s]
100%|████████████████████████████████████████| 235k/235k [00:00<00:00, 526kiB/s]


Загружена за 26.42 сек


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_asr.to(device)

for name, param in model_asr.named_parameters():
    if 'head' in name:
        param.requires_grad = True
    elif 'layers' in name:
        layer_num = int(name.split('.')[2])
        if layer_num >= 14:
            param.requires_grad = True
    else:
        param.requires_grad = False

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class ASRDataset(Dataset):
    def __init__(self, folder_path=None, txt_path=None, paths=None, labels=None):
        self.files = []

        if folder_path is not None and txt_path is not None:
          with open(txt_path, 'r', encoding='utf-8') as f:
              for line in f:
                  parts = line.strip().split(maxsplit=1)
                  if len(parts) == 2:
                      name, text = parts
                      wav_path = Path(folder_path) / f"{name}.wav"
                      if wav_path.exists():
                          self.files.append((str(wav_path), text))
        elif paths is not None and labels is not None:
            self.files = list(zip(paths, labels))
        else:
          raise ValueError("Нужно указать либо folder_path и txt_path, либо paths и labels")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, text = self.files[idx]
        return path, text

    @classmethod
    def from_lists(cls, paths, labels):
        return cls(paths=paths, labels=labels)


def collate_fn(batch):
    paths, texts = zip(*batch)
    return list(paths), list(texts)

dataset = ASRDataset(
    folder_path = "/content/drive/MyDrive/876/",
    txt_path = "/content/drive/MyDrive/876/Текст.txt"
)

In [5]:
from sklearn.model_selection import train_test_split

all_paths = [item[0] for item in dataset.files]
all_texts = [item[1] for item in dataset.files]

train_paths, temp_paths, train_texts, temp_texts = train_test_split(
    all_paths, all_texts, test_size=0.4, random_state=42
)

val_paths, test_paths, val_texts, test_texts = train_test_split(
    temp_paths, temp_texts, test_size=0.5, random_state=42
)

train_dataset = ASRDataset.from_lists(train_paths, train_texts)
val_dataset = ASRDataset.from_lists(val_paths, val_texts)
test_dataset = ASRDataset.from_lists(test_paths, test_texts)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [6]:
def tokenize_text(text, tokenizer):
    if tokenizer.charwise:
        char_to_id = {char: i for i, char in enumerate(tokenizer.vocab)}
        return [char_to_id.get(char, 0) for char in text]
    else:
        return tokenizer.model.encode(text)


tokenizer = model_asr.decoding.tokenizer
blank_id = model_asr.decoding.blank_id

In [ ]:
criterion = nn.CTCLoss(blank=blank_id, zero_infinity=True).to(device)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model_asr.parameters()), lr=1e-5)

best_val_loss = float('inf')
best_model_state = None

for epoch in range(20):
    model_asr.train()
    train_loss = 0
    train_batches = 0

    for paths, texts in train_loader:
        optimizer.zero_grad()

        batch_log_probs = []
        batch_targets = []
        batch_input_lengths = []
        batch_target_lengths = []

        for path, text in zip(paths, texts):
            log_probs = model_asr.get_logits(path).to(device)

            if torch.isnan(log_probs).any():
                print(f"⚠️ NaN в {path}, пропускаем")
                continue

            batch_log_probs.append(log_probs.squeeze(0))
            batch_input_lengths.append(log_probs.shape[1])

            tokens = tokenize_text(text, tokenizer)
            batch_targets.append(torch.tensor(tokens).to(device))
            batch_target_lengths.append(len(tokens))

        if len(batch_log_probs) == 0:
            continue

        max_input_len = max(batch_input_lengths)
        log_probs_padded = torch.stack([
            F.pad(lp, (0, 0, 0, max_input_len - lp.shape[0]))
            for lp in batch_log_probs
        ])

        max_target_len = max(batch_target_lengths)
        targets_padded = torch.stack([
            F.pad(t, (0, max_target_len - len(t)), value=blank_id)
            for t in batch_targets
        ])

        log_probs_ctc = log_probs_padded.transpose(0, 1)

        loss = criterion(
            log_probs_ctc,
            targets_padded,
            torch.tensor(batch_input_lengths).to(device),
            torch.tensor(batch_target_lengths).to(device)
        )

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / train_batches if train_batches > 0 else 0

    model_asr.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for paths, texts in val_loader:
            batch_log_probs = []
            batch_targets = []
            batch_input_lengths = []
            batch_target_lengths = []

            for path, text in zip(paths, texts):
                log_probs = model_asr.get_logits(path).to(device)
                if torch.isnan(log_probs).any():
                    continue
                batch_log_probs.append(log_probs.squeeze(0))
                batch_input_lengths.append(log_probs.shape[1])

                tokens = tokenize_text(text, tokenizer)
                batch_targets.append(torch.tensor(tokens).to(device))
                batch_target_lengths.append(len(tokens))

            if len(batch_log_probs) == 0:
                continue

            max_input_len = max(batch_input_lengths)
            log_probs_padded = torch.stack([
                F.pad(lp, (0, 0, 0, max_input_len - lp.shape[0]))
                for lp in batch_log_probs
            ])

            max_target_len = max(batch_target_lengths)
            targets_padded = torch.stack([
                F.pad(t, (0, max_target_len - len(t)), value=blank_id)
                for t in batch_targets
            ])

            log_probs_ctc = log_probs_padded.transpose(0, 1)

            loss = criterion(
                log_probs_ctc,
                targets_padded,
                torch.tensor(batch_input_lengths).to(device),
                torch.tensor(batch_target_lengths).to(device)
            )

            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches if val_batches > 0 else 0

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_state = model_asr.state_dict().copy()
        print(f"Лучшая модель (Val Loss: {best_val_loss:.4f})")

    print(f"Эпоха {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

Лучшая модель (Val Loss: 2.0407)
Эпоха 1 | Train Loss: 3.0301 | Val Loss: 2.0407
Лучшая модель (Val Loss: 1.5841)
Эпоха 2 | Train Loss: 1.7555 | Val Loss: 1.5841
Лучшая модель (Val Loss: 1.4427)
Эпоха 3 | Train Loss: 1.3724 | Val Loss: 1.4427
Лучшая модель (Val Loss: 1.3220)
Эпоха 4 | Train Loss: 1.1533 | Val Loss: 1.3220
Лучшая модель (Val Loss: 1.2544)
Эпоха 5 | Train Loss: 0.9745 | Val Loss: 1.2544
Лучшая модель (Val Loss: 1.1812)
Эпоха 6 | Train Loss: 0.8504 | Val Loss: 1.1812
Лучшая модель (Val Loss: 1.1431)
Эпоха 7 | Train Loss: 0.7239 | Val Loss: 1.1431
Лучшая модель (Val Loss: 1.1052)
Эпоха 8 | Train Loss: 0.6188 | Val Loss: 1.1052
Эпоха 9 | Train Loss: 0.5653 | Val Loss: 1.1716
Лучшая модель (Val Loss: 1.0764)
Эпоха 10 | Train Loss: 0.6399 | Val Loss: 1.0764
Эпоха 11 | Train Loss: 0.4541 | Val Loss: 1.0845
Эпоха 12 | Train Loss: 0.3963 | Val Loss: 1.1010
Эпоха 13 | Train Loss: 0.3477 | Val Loss: 1.1288
Эпоха 14 | Train Loss: 0.3150 | Val Loss: 1.1658
Эпоха 15 | Train Loss: 0.2

In [ ]:
torch.save(best_model_state, "/content/drive/MyDrive/asr_model_weights.pth")

In [ ]:
# Загружаем потом (сначала создаём такую же модель!)
#model_asr = gigaam.load_model("e2e_ctc")
model_asr.load_state_dict(torch.load("/content/drive/MyDrive/asr_model_weights.pth"))

In [7]:
pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.7 MB/s eta 0:00:00


In [15]:
model_asr.eval()

test_loss = 0
test_batches = 0

# Для WER/CER понадобятся списки
all_pred_texts = []
all_true_texts = []

with torch.no_grad():
    for paths, texts in test_loader:

        # Собираем выходы энкодера для loss
        for path, text in zip(paths, texts):
            # Получаем encoded (признаки) и логиты (для loss)
            wav, length = model_asr.prepare_wav(path)
            features, feature_lengths = model_asr.preprocessor(wav.float(), length)
            encoded, encoded_len = model_asr.encoder(features.float(), feature_lengths)

            # Для декодирования
            pred_text = model_asr.transcribe(path)
            all_pred_texts.append(pred_text)
            all_true_texts.append(text)


# Вычисляем WER
try:
    from jiwer import wer, cer
    wer_score = wer(all_true_texts, all_pred_texts)
    cer_score = cer(all_true_texts, all_pred_texts)
    print(f"Word Error Rate (WER): {wer_score:.4f}")
    print(f"Character Error Rate (CER): {cer_score:.4f}")
except ImportError:
    print("Для расчёта WER/CER установите: pip install jiwer")

# Показываем примеры
print("\nПримеры распознавания:")
for i in range(min(15, len(all_pred_texts))):
    print(f"True: {all_true_texts[i]}")
    print(f"Pred: {all_pred_texts[i]}")
    print()

Word Error Rate (WER): 0.4579
Character Error Rate (CER): 0.2052

Примеры распознавания:
True: не не не не, лучше не надо.
Pred: не-не-не-не, лучше не надо.

True: пошел отсюда, джой!
Pred: пошел отсюда, жой!

True: наливай что-нибудь крепкого.
Pred: наливай что-нибудь крепкое.

True: ты если щас шляпу не снимешь, я тя уничтожу. понял?
Pred: ты если щас шляпу не снимешь, я тебя уничтожу. понял?

True: {неразборчиво}
Pred:  ⁇ неразштыбочиво ⁇ 

True: прости, пожалуйста.
Pred: прости, пожалуйста.

True: уверены? уверены?!
Pred: уверенная! уверенная!

True: я не могу, извини.
Pred: я не могу, извини.

True: ты очень хороший.
Pred: очень хороший.

True: а как потом этого попа {запинка} задушили они, ну, вообще?
Pred: как потом этого попа задушили? они, ну, вообще.

True: ну, да!
Pred: ну да.

True: ну, вы, вы ехали назад. я вас из-за угла не {запинка} могла увидеть. я ехала вперёд. всё нормально.
Pred: ну, вы вы ехали назад. я вас из-за угла не могла увидеть. я ехала вперёд, всё нормально.

In [ ]:
audio_path = "/content/drive/MyDrive/testMySystem/command_anger_020.wav"
text = model_asr.transcribe(audio_path)
print(f"Распознано: {text}")

Распознано: Ну вы на меня ещё в суд подаёте.


In [ ]:
print(f"Тип токенизатора: {type(tokenizer)}")
print(f"charwise: {tokenizer.charwise}")
if tokenizer.charwise:
    print(f"vocab первых 10 символов: {tokenizer.vocab[:10]}")
else:
    print(f"model path: {tokenizer.model}")

Тип токенизатора: <class 'gigaam.decoding.Tokenizer'>
charwise: False
model path: <sentencepiece.SentencePieceProcessor; proxy of <Swig Object of type 'sentencepiece::SentencePieceProcessor *' at 0x7c52d8ea21f0> >


In [ ]:
num_layers = len(model_asr.encoder.layers)
print(f'Всего слоёв в энкодере: {num_layers}')

print("\nПоследние слои модели:")
for name, _ in list(model_asr.named_parameters())[-20:]:
    print(name)

print(f"\nДоступные слои для разморозки:")
for name, _ in model_asr.named_parameters():
    if 'layer' in name.lower():
        parts = name.split('.')
        for p in parts:
            if p.isdigit():
                print(f'  - {name}')
                break

Всего слоёв в энкодере: 16

Последние слои модели:
encoder.layers.15.norm_self_att.weight
encoder.layers.15.norm_self_att.bias
encoder.layers.15.self_attn.linear_q.weight
encoder.layers.15.self_attn.linear_q.bias
encoder.layers.15.self_attn.linear_k.weight
encoder.layers.15.self_attn.linear_k.bias
encoder.layers.15.self_attn.linear_v.weight
encoder.layers.15.self_attn.linear_v.bias
encoder.layers.15.self_attn.linear_out.weight
encoder.layers.15.self_attn.linear_out.bias
encoder.layers.15.norm_feed_forward2.weight
encoder.layers.15.norm_feed_forward2.bias
encoder.layers.15.feed_forward2.linear1.weight
encoder.layers.15.feed_forward2.linear1.bias
encoder.layers.15.feed_forward2.linear2.weight
encoder.layers.15.feed_forward2.linear2.bias
encoder.layers.15.norm_out.weight
encoder.layers.15.norm_out.bias
head.decoder_layers.0.weight
head.decoder_layers.0.bias

Доступные слои для разморозки:
  - encoder.layers.0.norm_feed_forward1.weight
  - encoder.layers.0.norm_feed_forward1.bias
  - encod